In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray          # registers the .rio accessor on xarray
from shapely.geometry import mapping
from pathlib import Path
from datetime import datetime
 
warnings.filterwarnings("ignore", category=FutureWarning)

## CONFIGURATION

edit these paths and variable names to match your files

In [36]:
# --- Paths ---
INPUT_DIR = Path("data")
SHAPEFILE_PATH = INPUT_DIR / "batas_provinsi/Provinsi_Kemdagri.shp"
 
YEARS = range(2021, 2026)  # 2021-2025
 
# TerraClimate variable name inside the .nc (usually matches the filename label)
NC_VARIABLES = {
    "tmax": "tmax",
    "tmin": "tmin",
    # "ppt":  "ppt",
}
 
OUTPUT_DIR      = Path("output")
CLIPPED_NC_DIR  = OUTPUT_DIR / "clipped_nc"
ZONAL_STATS_DIR = OUTPUT_DIR / "zonal_stats"
 
# Column in shapefile with province names
# Common: "NAME_1" (GADM), "PROVINSI", "ADM1_EN", "name"
PROVINCE_COL = "nama_prop"
 
TERRACLIMATE_CRS = "EPSG:4326"
CRS          = "EPSG:4326"
COMPUTE_ZONAL_STATS = True

# Set False to skip saving per-province .nc files (saves disk + time)
SAVE_CLIPPED_NC = True

 ## HELPER FUNCTIONS

In [13]:
def sanitize_name(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", name).strip("_").lower()

def open_nc(var, year):
    """Open one TerraClimate file, standardize dims, return DataArray."""
    f = INPUT_DIR / f"TerraClimate_{var}_{year}.nc"
    ds = xr.open_dataset(str(f), engine="netcdf4")
    da = ds[var]
 
    rename = {}
    if "lon" in da.dims: rename["lon"] = "x"
    if "lat" in da.dims: rename["lat"] = "y"
    if "longitude" in da.dims: rename["longitude"] = "x"
    if "latitude" in da.dims: rename["latitude"] = "y"
    if rename:
        da = da.rename(rename)
 
    da = da.rio.write_crs(CRS)
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y")
    if da.y[0] < da.y[-1]:
        da = da.sortby("y", ascending=False)
    return da, ds

def load_and_concat_yearly(variable: str, var_name: str) -> xr.DataArray:
    """
    Load TerraClimate_<var>_<year>.nc for each year, concat along time.
    """
    arrays = []
    for year in YEARS:
        fpath = INPUT_DIR / f"TerraClimate_{variable}_{year}.nc"
        if not fpath.exists():
            print(f"    WARNING: {fpath} not found, skipping")
            continue
        ds = xr.open_dataset(str(fpath), engine="netcdf4")
        da = ds[var_name]
        print(f"    {fpath.name}: time={da.sizes.get('time', '?')}")
        arrays.append(da)

    if not arrays:
        raise FileNotFoundError(f"No files found for '{variable}'")

    da = xr.concat(arrays, dim="time")

    # Standardize dim names for rioxarray
    rename = {}
    if "lon" in da.dims:
        rename["lon"] = "x"
    if "lat" in da.dims:
        rename["lat"] = "y"
    if "longitude" in da.dims:
        rename["longitude"] = "x"
    if "latitude" in da.dims:
        rename["latitude"] = "y"
    if rename:
        da = da.rename(rename)

    da = da.rio.write_crs(TERRACLIMATE_CRS)
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y")

    if da.y[0] < da.y[-1]:
        da = da.sortby("y", ascending=False)

    return da


def clip_to_polygon(da: xr.DataArray, geometry) -> xr.DataArray:
    return da.rio.clip([mapping(geometry)], crs=TERRACLIMATE_CRS,
                       drop=True, all_touched=True)


def compute_stats(clipped: xr.DataArray, province: str, var_label: str):
    records = []
    for t in range(clipped.sizes["time"]):
        s = clipped.isel(time=t)
        tv = pd.Timestamp(s.time.values)
        vals = s.values.flatten()
        vals = vals[~np.isnan(vals)]
        n = len(vals)
        records.append({
            "province": province,
            "variable": var_label,
            "year": tv.year,
            "month": tv.month,
            "date": tv.strftime("%Y-%m"),
            "mean":   float(np.mean(vals))   if n else np.nan,
            "median": float(np.median(vals)) if n else np.nan,
            "min":    float(np.min(vals))    if n else np.nan,
            "max":    float(np.max(vals))    if n else np.nan,
            "std":    float(np.std(vals))    if n else np.nan,
            "pixel_count": n,
        })
    return records

def zonal_stats(clipped, province, var):
    """Extract mean/min/max/std per timestep."""
    rows = []
    for t in range(clipped.sizes["time"]):
        s = clipped.isel(time=t)
        tv = pd.Timestamp(s.time.values)
        v = s.values.flatten()
        v = v[~np.isnan(v)]
        n = len(v)
        rows.append({
            "province": province, "variable": var,
            "year": tv.year, "month": tv.month,
            "date": tv.strftime("%Y-%m"),
            "mean":   float(np.mean(v))   if n else np.nan,
            "median": float(np.median(v)) if n else np.nan,
            "min":    float(np.min(v))    if n else np.nan,
            "max":    float(np.max(v))    if n else np.nan,
            "std":    float(np.std(v))    if n else np.nan,
            "pixel_count": n,
        })
    return rows

In [31]:
t0 = datetime.now()
print("=" * 70)
print("TerraClimate Batch Clip - Indonesia Provinces")
print(f"Started : {t0:%Y-%m-%d %H:%M:%S}")
print(f"Python  : {os.sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"xarray  : {xr.__version__}")
print("=" * 70)

# --- Shapefile ---
gdf = gpd.read_file(SHAPEFILE_PATH)
print(f"\nShapefile: {len(gdf)} features")
print(f"Columns : {list(gdf.columns)}")

if PROVINCE_COL not in gdf.columns:
    print(f"\n*** '{PROVINCE_COL}' not found! Available: {list(gdf.columns)}")
    # return

if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)


TerraClimate Batch Clip - Indonesia Provinces
Started : 2026-04-06 13:20:25
Python  : 3.8.20
NumPy   : 1.24.3
xarray  : 2023.1.0

Shapefile: 38 features
Columns : ['objectid', 'no_prop', 'nama_prop', 'jumlah_kab', 'jumlah_kot', 'jumlah_kec', 'jumlah_des', 'jumlah_kel', 'jumlah_pen', 'jumlah_kk', 'luas_wilay', 'kepadatan_', 'perpindaha', 'jml_mening', 'jml_lahir', 'perubahan_', 'jml_wktp', 'jml_rekam_', 'field1', 'field2', 'islam', 'kristen', 'katholik', 'hindu', 'budha', 'konghucu', 'kepercayaa', 'field3', 'field4', 'pria', 'wanita', 'field5', 'field6', 'belum_kawi', 'kawin', 'cerai_hidu', 'cerai_mati', 'field7', 'field8', 'u0', 'u5', 'u10', 'u15', 'u20', 'u25', 'u30', 'u35', 'u40', 'u45', 'u50', 'u55', 'u60', 'u65', 'u70', 'u75', 'field9', 'field10', 'lhr_2020', 'lhr_sebelu', 'lhr_2021', 'lhr_sebe_1', 'lhr_2022', 'lhr_sebe_2', 'lhr_2023', 'lhr_sebe_3', 'lhr_2024', 'lhr_sebe_4', 'pertumbuha', 'pertumbu_1', 'pertumbu_2', 'pertumbu_3', 'pertumbu_4', 'field11', 'field12', 'pendidikan', 'p

In [12]:
import gc
gc.collect()

1853

In [11]:
# --- Load & concat yearly NCs ---
data_arrays = {}
for label, var_name in NC_VARIABLES.items():
    print(f"\nLoading {label}:")
    da = load_and_concat_yearly(label, var_name)
    print(f"  Concatenated: {da.shape} ({da.sizes['time']} months)")
    data_arrays[label] = da

# --- Prep output ---
CLIPPED_NC_DIR.mkdir(parents=True, exist_ok=True)
ZONAL_STATS_DIR.mkdir(parents=True, exist_ok=True)



Loading ppt:
    TerraClimate_ppt_2021.nc: time=12
    TerraClimate_ppt_2022.nc: time=12
    TerraClimate_ppt_2023.nc: time=12
    TerraClimate_ppt_2024.nc: time=12
    TerraClimate_ppt_2025.nc: time=12


MemoryError: Unable to allocate 3.34 GiB for an array with shape (12, 4320, 8640) and data type float64

In [ ]:

all_stats = []
n = len(gdf)

# --- Clip loop ---
for i, (_, row) in enumerate(gdf.iterrows(), 1):
    prov = row[PROVINCE_COL]
    safe = sanitize_name(prov)
    pdir = CLIPPED_NC_DIR / safe
    pdir.mkdir(parents=True, exist_ok=True)

    print(f"\n[{i}/{n}] {prov}")

    for label, da in data_arrays.items():
        out = pdir / f"{label}_{safe}.nc"
        try:
            clipped = clip_to_polygon(da, row.geometry)

            clipped_ds = clipped.to_dataset(name=label)
            clipped_ds.attrs.update({
                "province": prov,
                "source": "TerraClimate",
                "period": "2021-2025",
            })
            clipped_ds.to_netcdf(out)

            kb = out.stat().st_size / 1024
            print(f"  {label:4s} -> {clipped.shape}  ({kb:.0f} KB)")

            if COMPUTE_ZONAL_STATS:
                all_stats.extend(compute_stats(clipped, prov, label))

        except Exception as e:
            print(f"  {label} FAILED: {e}")

# --- Zonal stats CSV ---
if COMPUTE_ZONAL_STATS and all_stats:
    df = pd.DataFrame(all_stats)
    sp = ZONAL_STATS_DIR / "zonal_summary.csv"
    df.to_csv(sp, index=False)
    print(f"\nZonal stats -> {sp}")
    print(f"  {len(df)} records | {df['province'].nunique()} provinces")

print(f"\nDone in {(datetime.now() - t0).total_seconds():.1f}s")

In [19]:
t0 = datetime.now()
print("=" * 60)
print("TerraClimate Batch Clip (low-memory mode)")
print("=" * 60)

# Load shapefile (small, stays in memory the whole time)
gdf = gpd.read_file(SHAPEFILE_PATH)
if PROVINCE_COL not in gdf.columns:
    print(f"Column '{PROVINCE_COL}' not found. Available: {list(gdf.columns)}")
    #return

if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
print(f"Provinces: {len(gdf)}")

TerraClimate Batch Clip (low-memory mode)
Provinces: 38


In [37]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
all_stats = []

# Outer loop: one file at a time (3 vars x 5 years = 15 iterations)
total_files = len(NC_VARIABLES) * len(YEARS)
file_i = 0

In [40]:
for var in NC_VARIABLES:
    for year in YEARS:
        file_i += 1
        fname = f"TerraClimate_{var}_{year}.nc"
        fpath = INPUT_DIR / fname

        if not fpath.exists():
            print(f"[{file_i}/{total_files}] {fname} — NOT FOUND, skip")
            continue

        print(f"\n[{file_i}/{total_files}] {fname}")
        da, ds = open_nc(var, year)

        # Clip each province against this single file
        for _, row in gdf.iterrows():
            prov = row[PROVINCE_COL]
            safe = sanitize_name(prov)

            try:
                clipped = da.rio.clip(
                    [mapping(row.geometry)], crs=CRS,
                    drop=True, all_touched=True
                )

                # Collect stats
                all_stats.extend(zonal_stats(clipped, prov, var))

                # Optionally save clipped .nc
                if SAVE_CLIPPED_NC:
                    pdir = OUTPUT_DIR / "clipped_nc" / safe
                    pdir.mkdir(parents=True, exist_ok=True)
                    out = pdir / f"{var}_{year}_{safe}.nc"
                    save_ds = clipped.to_dataset(name=var)
                    # Remove grid_mapping to prevent encoding conflict
                    for v in save_ds.data_vars:
                        save_ds[v].attrs.pop("grid_mapping", None)
                    save_ds.attrs.pop("grid_mapping", None)
                    save_ds.to_netcdf(out)

                del clipped

            except Exception as e:
                print(f"  {prov}: {e}")

        # Close and free memory before loading next file
        ds.close()
        del da, ds
        gc.collect()
        print(f"  done — {len(gdf)} provinces clipped")


[1/10] TerraClimate_tmax_2021.nc
  done — 38 provinces clipped

[2/10] TerraClimate_tmax_2022.nc
  done — 38 provinces clipped

[3/10] TerraClimate_tmax_2023.nc
  done — 38 provinces clipped

[4/10] TerraClimate_tmax_2024.nc
  done — 38 provinces clipped

[5/10] TerraClimate_tmax_2025.nc
  done — 38 provinces clipped

[6/10] TerraClimate_tmin_2021.nc
  done — 38 provinces clipped

[7/10] TerraClimate_tmin_2022.nc
  done — 38 provinces clipped

[8/10] TerraClimate_tmin_2023.nc
  done — 38 provinces clipped

[9/10] TerraClimate_tmin_2024.nc
  done — 38 provinces clipped

[10/10] TerraClimate_tmin_2025.nc
  done — 38 provinces clipped


In [1]:
import geopandas as gpd
import pandas as pd

# 1. Define your file paths
shapefile_path = "data/batas_kabupaten_kota/Batas Kabupaten Kemendagri 2024.shp"
pickle_path = "data/batas_kabupaten_kota.pkl"

# 2. Load the shapefile into a GeoDataFrame
# Note: Ensure the .shx and .dbf files are in the same folder as your .shp file
print("Loading shapefile...")
gdf = gpd.read_file(shapefile_path)

# 3. Save (export) the GeoDataFrame to a pickle file
print("Saving to pickle...")
gdf.to_pickle(pickle_path)

print("Done!")

# --- Optional: How to read the pickle file back later ---
# loaded_gdf = pd.read_pickle(pickle_path)
# print(loaded_gdf.head())

Loading shapefile...
Saving to pickle...
Done!


In [2]:
import geopandas as gpd

# Load the shapefile
shapefile_path = "data/batas_kabupaten_kota/Batas Kabupaten Kemendagri 2024.shp"
gdf = gpd.read_file(shapefile_path)

# Save to Parquet (defaults to 'snappy' compression)
parquet_path = "data/batas_kabupaten_kota.parquet"
gdf.to_parquet(parquet_path)

print("Saved to Parquet!")

Saved to Parquet!


In [10]:
# show all columns in head
pd.set_option('display.max_columns', None)
gdf.head()

,objectid,nama_prop,nama_kab,no_prop,no_kab,kode_prop_,kode_kab_s,jumlah_des,jumlah_kel,jumlah_kec,jumlah_pen,jumlah_kk,luas_wilay,kepadatan_,perpindaha,jml_mening,jml_lahir,perubahan_,jml_wktp,jml_rekam_,field1,field2,islam,kristen,katholik,hindu,budha,konghucu,kepercayaa,field3,field4,pria,wanita,field5,field6,belum_kawi,kawin,cerai_hidu,cerai_mati,field7,field8,u0,u5,u10,u15,u20,u25,u30,u35,u40,u45,u50,u55,u60,u65,u70,u75,field9,field10,lhr_2020,lhr_sebelu,lhr_2021,lhr_sebe_1,lhr_2022,lhr_sebe_2,lhr_2023,lhr_sebe_3,lhr_2024,lhr_sebe_4,pertumbuha,pertumbu_1,pertumbu_2,pertumbu_3,pertumbu_4,field11,field12,pendidikan,pendidik_1,pendidik_2,pendidik_3,pendidik_4,pendidik_5,field13,field14,tidak_blm_,belum_tama,tamat_sd,sltp,slta,d1_dan_d2,d3,s1,s2,s3,field15,field16,o,b_,b1,o_,ab_,a,tidak_tahu,ab1,a_,a1,o1,b,ab,field17,field18,pensiunan,mengurus_r,belum_tida,lainnya,perdaganga,perawat,nelayan,pelajar_ma,guru,wiraswasta,pengacara,field19,field20,shape_Leng,shape_Area,geometry
0,1.0,ACEH,KAB. ACEH BARAT,11.0,5.0,11.0,1105.0,321.0,NaN,12.0,207690.0,66150.0,2782.873,74.631505,348.0,65.0,207690.0,189642.0,148981.0,145250.0,-,-,206537.0,412.0,69.0,0.0,672.0,0.0,0.0,-,-,104626.0,103064.0,-,-,98617.0,96059.0,2723.0,10291.0,-,-,14577.0,18251.0,18450.0,17895.0,16236.0,16282.0,15582.0,16094.0,16112.0,14268.0,13010.0,10163.0,7912.0,5053.0,3535.0,4270.0,-,-,3579.0,193113.0,3733.0,196692.0,3579.0,200425.0,3733.0,203869.0,3733.0,206933.0,2.0,2.0,2.0,2.0,0.0,-,-,7312.0,3551.0,22013.0,11137.0,11278.0,15975.0,-,-,37887.0,25127.0,39761.0,28627.0,57095.0,2268.0,3861.0,12267.0,768.0,29.0,-,-,13166.0,15.0,160.0,122.0,32.0,4661.0,181166.0,52.0,149.0,17.0,126.0,5974.0,2050.0,-,-,1528.0,45596.0,37515.0,3.0,459.0,243.0,2502.0,55976.0,1056.0,19821.0,2.0,-,-,2.624683,0.226751,"POLYGON Z ((96.26028 4.71238 0.00000, 96.25721..."
1,2.0,ACEH,KAB. ACEH BARAT DAYA,11.0,12.0,11.0,1112.0,152.0,NaN,9.0,154997.0,47224.0,1882.277,82.345478,272.0,49.0,154997.0,137530.0,109843.0,106589.0,-,-,154740.0,42.0,4.0,12.0,199.0,0.0,0.0,-,-,78086.0,76911.0,-,-,76961.0,67390.0,1494.0,9152.0,-,-,11688.0,14325.0,13659.0,12869.0,13259.0,12729.0,11950.0,12085.0,12056.0,10510.0,8971.0,7248.0,5689.0,3306.0,2243.0,2410.0,-,-,2908.0,143309.0,2964.0,146217.0,2908.0,149181.0,2964.0,151943.0,2964.0,154346.0,2.0,2.0,2.0,2.0,0.0,-,-,5872.0,2846.0,16876.0,8262.0,8142.0,12235.0,-,-,41814.0,16516.0,33841.0,23204.0,29419.0,1411.0,2238.0,6330.0,217.0,7.0,-,-,4505.0,24.0,24.0,92.0,43.0,1635.0,146094.0,21.0,24.0,9.0,316.0,1713.0,497.0,-,-,679.0,37082.0,40704.0,1.0,194.0,163.0,2658.0,32699.0,751.0,16658.0,11.0,-,-,2.272465,0.153250,"MULTIPOLYGON Z (((96.72918 4.09413 0.00000, 96..."
2,3.0,ACEH,KAB. ACEH BESAR,11.0,6.0,11.0,1106.0,604.0,NaN,23.0,439048.0,130573.0,2891.477,151.842121,678.0,138.0,439048.0,398019.0,303804.0,299982.0,-,-,437863.0,847.0,194.0,12.0,132.0,0.0,0.0,-,-,220028.0,219020.0,-,-,223585.0,188973.0,4598.0,21892.0,-,-,32790.0,42696.0,42291.0,35803.0,35053.0,33367.0,33757.0,35631.0,34140.0,28557.0,25007.0,19239.0,14759.0,9518.0,7117.0,9323.0,-,-,8417.0,406258.0,8489.0,414675.0,8417.0,423164.0,8489.0,430651.0,8489.0,437393.0,2.0,2.0,2.0,2.0,0.0,-,-,16906.0,8320.0,51060.0,25607.0,22564.0,34421.0,-,-,90841.0,52853.0,59058.0,61763.0,123850.0,4631.0,11035.0,34109.0,858.0,50.0,-,-,35746.0,62.0,665.0,731.0,84.0,11909.0,369035.0,222.0,604.0,53.0,398.0,14858.0,4681.0,-,-,3755.0,87115.0,94598.0,1.0,491.0,624.0,2405.0,118636.0,3311.0,25071.0,44.0,-,-,4.668149,0.235919,"MULTIPOLYGON Z (((95.43359 5.65658 0.00000, 95..."
3,4.0,ACEH,KAB. ACEH JAYA,11.0,14.0,11.0,1114.0,172.0,NaN,9.0,100674.0,31407.0,3872.352,25.998153,176.0,37.0,100674.0,94154.0,70256.0,68084.0,-,-,100605.0,50.0,13.0,0.0,6.0,0.0,0.0,-,-,51030.0,49644.0,-,-,50118.0,44266.0,1248.0,5042.0,-,-,7303.0,9565.0,9620.0,9711.0,7235.0,7383.0,7585.0,8388.0,8268.0,6603.0,5577.0,4283.0,3344.0,1973.0,1335.0,2501.0,-,-,1834.0,93371.0,1803.0,95205.0,1834.0,97008.0,1803.0,98649.0,1803.0,100180.0,2.0,2.0,2.0,2.0,0.0,-,-,3637.0,1879.0,11511.0,5795.0,

In [ ]:
gdf["geometry"] = gdf.geometry.simplify(0.005, preserve_topology=True)

# Keep only essential columns (adjust names if different)
keep = [c for c in gdf.columns if c.lower() in 
        ("nama_prop","nama_kab","no_prop","no_kab","geometry")]
if "geometry" not in keep:
    keep.append("geometry")
gdf_light = gdf[keep]

gdf_light.to_parquet("data/kabupaten_simple.parquet")
print(f"Done! {gdf_light.memory_usage(deep=True).sum()/1e6:.1f} MB in memory")

Done! 0.1 MB in memory


In [12]:
print(gdf.columns.tolist())

['objectid', 'nama_prop', 'nama_kab', 'no_prop', 'no_kab', 'kode_prop_', 'kode_kab_s', 'jumlah_des', 'jumlah_kel', 'jumlah_kec', 'jumlah_pen', 'jumlah_kk', 'luas_wilay', 'kepadatan_', 'perpindaha', 'jml_mening', 'jml_lahir', 'perubahan_', 'jml_wktp', 'jml_rekam_', 'field1', 'field2', 'islam', 'kristen', 'katholik', 'hindu', 'budha', 'konghucu', 'kepercayaa', 'field3', 'field4', 'pria', 'wanita', 'field5', 'field6', 'belum_kawi', 'kawin', 'cerai_hidu', 'cerai_mati', 'field7', 'field8', 'u0', 'u5', 'u10', 'u15', 'u20', 'u25', 'u30', 'u35', 'u40', 'u45', 'u50', 'u55', 'u60', 'u65', 'u70', 'u75', 'field9', 'field10', 'lhr_2020', 'lhr_sebelu', 'lhr_2021', 'lhr_sebe_1', 'lhr_2022', 'lhr_sebe_2', 'lhr_2023', 'lhr_sebe_3', 'lhr_2024', 'lhr_sebe_4', 'pertumbuha', 'pertumbu_1', 'pertumbu_2', 'pertumbu_3', 'pertumbu_4', 'field11', 'field12', 'pendidikan', 'pendidik_1', 'pendidik_2', 'pendidik_3', 'pendidik_4', 'pendidik_5', 'field13', 'field14', 'tidak_blm_', 'belum_tama', 'tamat_sd', 'sltp', 'sl

In [19]:
shp_path = "data/batas_provinsi/Provinsi_Kemdagri.shp"
gdf_prov = gpd.read_file(shp_path)
gdf_prov = gdf_prov.to_crs("EPSG:4326")
gdf_prov["geometry"] = gdf_prov.geometry.simplify(0.005, preserve_topology=True)
gdf_prov[["nama_prop", "geometry"]].to_file(
"data/provinces_3.geojson", driver="GeoJSON"
)